# Convolutions


## Definition: Infinite 2D Convolution

Input  
* $X \in \mathbb{R}^{C_1\times \mathbb{Z}^2}.$  

Weights  
* kernel $W \in \mathbb{R}^{C_2\times C_1 \times h \times w}$  
* bias $b \in \mathbb{R}^{C_2}$  

Hyperparameters  
* stride $s=(s_1,s_2)$    

Output  
* $\operatorname{InfConv2d}_{W,b}(X) = O \in \mathbb{R}^{C_2\times \mathbb{Z}^2}$, where  

$$
\begin{align*}
O_{i,j,k} &= b_{i} + \sum_{l=1}^{C_1} \sum_{m=1}^{h} \sum_{n=1}^{w}
x_{l,\,(j-1)\cdot s_1 + m,\, (k-1)\cdot s_2 + n}\,
\omega_{i,l,m,n}.
\end{align*}
$$


### Property: $\operatorname{InfConv2d}_{W,0}$ is linear and $\operatorname{InfConv2d}_{W,b}$ is affine

Proof:
$$
\begin{align*}
\operatorname{InfConv2d}_{W,0}(\alpha X+\beta Y) &=\sum_{l=1}^{C_1}\sum_{m=1}^{h}\sum_{n=1}^{w}(\alpha x_{l,(j-1)s_1+m,(k-1)s_2+n}+\beta y_{l,(j-1)s_1+m,(k-1)s_2+n})\cdot \omega_{i,l,m,n},\\
&=\sum_{l=1}^{C_1}\sum_{m=1}^{h}\sum_{n=1}^{w}(\alpha x_{l,(j-1)s_1+m,(k-1)s_2+n}\cdot \omega_{i,l,m,n}+\beta y_{l,(j-1)s_1+m,(k-1)s_2+n}\cdot \omega_{i,l,m,n}),\\
&=\alpha\sum_{l=1}^{C_1}\sum_{m=1}^{h}\sum_{n=1}^{w} x_{l,(j-1)s_1+m,(k-1)s_2+n}\cdot \omega_{i,l,m,n}+\beta \sum_{l=1}^{C_1}\sum_{m=1}^{h}\sum_{n=1}^{w}y_{l,(j-1)s_1+m,(k-1)s_2+n}\cdot \omega_{i,l,m,n},\\
&=\alpha \operatorname{InfConv2d}_{W,0}(X)+\beta \operatorname{InfConv2d}_{W,0}(Y).
\end{align*}
$$

#### Definition: Positional translation
Given $\delta\in \mathbb{Z}^2$ and a channel dimension $C\in\mathbb{N}$. We define the "positional" translation of $\delta$ units as
$$
\begin{align*}
T^{C}_{\delta}:  \mathbb{R}^{C\times \mathbb{Z}^2}&\longmapsto \mathbb{R}^{C\times \mathbb{Z}^2} ,&\quad \text{where} \quad   Y_{i,j,k}&=X_{i,j+\delta_1,k+\delta_2}.\\
 T^{C}_{\delta} (X) &= Y & 
\end{align*}
$$

### Property: $\operatorname{InfConv2d}_{W,b}$ is translation-equivariant up to stride

If $\operatorname{InfConv2d}_{W,b}: \mathbb{R}^{C_1\times \mathbb{Z}^2} \longmapsto \mathbb{R}^{C_2\times \mathbb{Z}^2}$ with stride $s$, then
$$\operatorname{InfConv2d}_{W,b}\circ \ T^{C_1}_{s*\delta} = T^{C_2}_{\delta}\circ \operatorname{InfConv2d}_{W,b}, \quad \forall\delta \in \mathbb{Z}^2.$$

In particular, for stride $s=(1,1)$, **full translation equivariance** holds.


Proof: Let $(i,j,k)\in C\times\mathbb{Z}^2$ 
$$
\begin{align*}
\operatorname{InfConv2d}_{W,b}\circ \ T^{C_1}_{s*\delta}(X)_{i,j,k}&=\operatorname{InfConv2d}_{W,b}\left( T^{C_1}_{s*\delta}(X)\right)_{i,j,k}\omega_{i,l,m,n},\\
&=b_{i} + \sum_{l=1}^{C_1} \sum_{m=1}^{h} \sum_{n=1}^{w}
T^{C_1}_{s*\delta}(X)_{l,\,(j-1)\cdot s_1 + m,\, (k-1)\cdot s_2 + n}\omega_{i,l,m,n},\\
&=b_{i} + \sum_{l=1}^{C_1} \sum_{m=1}^{h} \sum_{n=1}^{w}
x_{l,\,(j-1)\cdot s_1 + m+s_1\delta_1,\, (k-1)\cdot s_2 + n+s_2\delta_2}\omega_{i,l,m,n},\\
&=b_{i} + \sum_{l=1}^{C_1} \sum_{m=1}^{h} \sum_{n=1}^{w}
x_{l,\,(j+\delta_1-1)\cdot s_1 + m,\, (k+\delta_2-1)\cdot s_2 + n}\omega_{i,l,m,n},\\
&=\operatorname{InfConv2d}_{W,b}(X)_{i,j+\delta_1,k+\delta_2},\\
&=T^{C_2}_{\delta}(\operatorname{InfConv2d}_{W,b}(X))_{i,j,k},\\
&=T^{C_2}_{\delta}\circ\operatorname{InfConv2d}_{W,b}(X)_{i,j,k}.
\end{align*}
$$

Then $\operatorname{Conv2d}_{W,b}\circ \ T^{C_1}_{s*\delta} = T^{C_2}_{\delta}\circ \operatorname{Conv2d}_{W,b}$ and we conclude that $\operatorname{InfConv2d}_{W,b}$ is translation equivariance


#### Definition: Padding operation

$$
\begin{align*}
\operatorname{Pad}^{(p_1,p_2)}:\bigcup_{H,W \in \mathbb{N}}\mathbb{R}^{C\times H\times W}&\longmapsto \mathbb{R}^{C\times \mathbb{Z}^2}, \\
\operatorname{Pad}^{(p_1,p_2)} (X) &= Y
\end{align*}
$$
where
$$
Y_{i,j,k}=\begin{cases} X_{i,j-p_1,k-p_2}, & p_1< j\leq H+p_1, p_2 < k\leq W+p_2, \\
0,& \text{otherwise} \\
\end{cases}\\
$$

### Property: $\operatorname{Pad}^{(p_1,p_2)}\big|_{\mathbb{R}^{C\times H\times W}}$ is linear 

Proof: Let $\alpha,\beta \in \mathbb{R}$, $X,Y \in \mathbb{R}^{C\times H\times W}$, and $(i,j,k)\in C\times \mathbb{Z}^2$ then 
$$
\begin{align*}
\operatorname{Pad}^{(p_1,p_2)} (\alpha X+ \beta Y)_{i,j,k}&=\begin{cases} \alpha X_{i,j-p_1,k-p_2}+\beta Y_{i,j-p_1,k-p_2}, & p_1< j\leq H+p_1, p_2 < k\leq W+p_2 \\
0,& \text{otherwise} \\
\end{cases}\\
&=\alpha\begin{cases} X_{i,j-p_1,k-p_2}, & p_1< j\leq H+p_1, p_2 < k\leq W+p_2 \\
0,& \text{otherwise} \\
\end{cases}\\
&+\beta \begin{cases}Y_{i,j-p_1,k-p_2}, & p_1< j\leq H+p_1, p_2 < k\leq W+p_2 \\
0,& \text{otherwise} \\
\end{cases}\\
&=\alpha\operatorname{Pad}^{(p_1,p_2)} (X)_{i,j,k}+\beta \operatorname{Pad}^{(p_1,p_2)} (Y)_{i,j,k},\\
&=(\alpha\operatorname{Pad}^{(p_1,p_2)} (X)+\beta \operatorname{Pad}^{(p_1,p_2)} (Y))_{i,j,k},\\
\end{align*}
$$
 
 Then $\operatorname{Pad}^{(p_1,p_2)} (\alpha X+ \beta Y) = \alpha\operatorname{Pad}^{(p_1,p_2)} (X)+\beta \operatorname{Pad}^{(p_1,p_2)} (Y)$ and we conclude that $\operatorname{Pad}^{(p_1,p_2)}\big|_{\mathbb{R}^{C\times H\times W}}$ is linear

#### Definition: Crop operation

$$
\begin{align*}
\operatorname{Crop}^{(*,H,W)}\!: \ \ \mathbb{R}^{C\times\mathbb{Z}^2}&\longmapsto \mathbb{R}^{C\times H\times W} ,\\
\operatorname{Crop}^{(*,H,W)} (X) &= Y
\end{align*}
$$

where
$$
Y_{i,j,k}=X_{i,j,k}, \quad  1\leq j\leq H, 1\leq k\leq W.\\
$$

### Property: $\operatorname{Crop}^{(*,H,W)}$ is linear 

Proof: Let $\alpha,\beta \in \mathbb{R}$, $X,Y \in \mathbb{R}^{C\times H\times W}$, and $(i,j,k)\in C\times H \times W$ then 
$$
\begin{align*}
\operatorname{Crop}^{(*,H,W)}(\alpha X+ \beta Y)_{i,j,k}&=(\alpha X+\beta Y)_{i,j,k}, \\
&=\alpha X_{i,j,k}+\beta Y_{i,j,k}, \\
&=\alpha\operatorname{Crop}^{(*,H,W)} (X)_{i,j,k}+\beta \operatorname{Crop}^{(*,H,W)} (Y)_{i,j,k},\\
&=(\alpha\operatorname{Crop}^{(*,H,W)} (X)+\beta \operatorname{Crop}^{(*,H,W)} (Y))_{i,j,k},\\
\end{align*}
$$
 
 Then $\operatorname{Crop}^{(*,H,W)}(\alpha X+ \beta Y) = \alpha\operatorname{Crop}^{(*,H,W)}(X)+\beta \operatorname{Crop}^{(*,H,W)} (Y)$ and we conclude that $\operatorname{Crop}^{(*,H,W)}$ is linear

## Definition: Finite 2D Convolution (with padding)

In practice we don't have "inifinite" tensors, we only can pass finite dimensional inputs to convolution, so we need a finite convolution definition

Input  
* $X \in \mathbb{R}^{C_1\times H \times W}, \quad \text{for } H,W\in \mathbb{N}.$  

Weights  
* kernel $W \in \mathbb{R}^{C_2\times C_1 \times h \times w}$  
* bias $b \in \mathbb{R}^{C_2}$  

Hyperparameters  
* stride $s=(s_1,s_2)$  
* padding $p=(p_1,p_2)$  

Output  
* $\operatorname{Conv2d}_{W,b} = \operatorname{Crop}^{(*,H',W')}\circ\operatorname{InfConv2d}_{W,b}\circ\operatorname{Pad}^{(p_1,p_2)}$ 

where  
$$
\begin{align*}
H' &= \left\lfloor \frac{H + 2p_1 - h}{s_1} \right\rfloor + 1, \\
W' &= \left\lfloor \frac{W + 2p_2 - w}{s_2} \right\rfloor + 1.
\end{align*}
$$

Notice that the indexes position $(j,k): j<1,j>H+2p_1, k<1$, or $k>W+2p_2$ are created by the padding and then deleted by the Crop. So we can simplified the formulation as follow   

Output  
* $\operatorname{Conv2d}_{W,b}(X) = O \in \mathbb{R}^{C_2 \times H' \times W'}$, where  

$$
\begin{align*}
O_{i,j,k} &= b_{i} + \sum_{l=1}^{C_1} \sum_{m=1}^{h} \sum_{n=1}^{w}
\tilde{x}_{l,\,(j-1)\cdot s_1 + m,\, (k-1)\cdot s_2 + n}\,
\omega_{i,l,m,n}.
\end{align*}
$$

where $\tilde{X} \in \mathbb{R}^{C_1 \times (H + 2p_1) \times (W + 2p_2)}$ is $X$ padded with $p_1$ zeros on the top and bottom and $p_2$ zeros on the left and right for each channel: 
$$
\begin{align*}
\operatorname{Pad}(X)&=\tilde{X},\\
\tilde{X}_{i,:,:} & = \begin{pmatrix} 
0_{p_1\!\times p_2} & 0_{p_1\!\times W} & 0_{p_1\!\times p_2},\\
0_{H \!\times p_2} & X_{i,:,:} & 0_{H \!\times p_2},\\
0_{p_1\!\times p_2} & 0_{p_1\!\times W} & 0_{p_1\!\times p_2},\\
\end{pmatrix}, \quad i=1,2,\dots,C_1.
\end{align*}
$$

**Note:** The same convolution function can be applied to inputs of different spatial shapes. The model parameters are not tied to specific coordinates. In this sense, $\operatorname{Conv2d}_{W,b}$ is **position-agnostic**.

### Property: $\operatorname{Conv2d}_{W,0}\big|_{\mathbb{R}^{C\times H\times W}}$ is linear and $\operatorname{Conv2d}_{W,b}\big|_{\mathbb{R}^{C\times H\times W}}$ is affine 

Proof: Notice that 
$$
\begin{align*}
\operatorname{Conv2d}_{W,b}\big|_{\mathbb{R}^{C\times H\times W}}=\operatorname{Crop}^{(*,H',W')}\circ\operatorname{InfConv2d}_{W,b}\circ(\operatorname{Pad}^{(p_1,p_2)}\big|_{\mathbb{R}^{C\times H\times W}})
\end{align*}
$$
Then the results comes directly from the previous results: 
* $\operatorname{Crop}^{(*,H',W')}$ is linear, 
* $\operatorname{Pad}^{(p_1,p_2)}\big|_{\mathbb{R}^{C\times H\times W}}$ is linear  
* $\operatorname{InfConv2d}_{W,0}$ is linear and $\operatorname{InfConv2d}_{W,b}$ is affine

## Code: $\operatorname{Conv2d}_{W,b}$

$\operatorname{InfConv2d}_{W,b}$ requires infinite dimentional inputs so it has only theoretical meaning

In [ ]:
import conv2d as mynn
import torch
import torch.nn as nn

## Testing

In [ ]:
in_channels = 3
out_channels = 2
kernel_size = 3
stride = 1
padding = 1

batch_size = 2

In [ ]:
torch.manual_seed(0)
nn_conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding)
torch.manual_seed(0)
conv = mynn.Conv2d(in_channels, out_channels, kernel_size, stride, padding)

## Weights

In [ ]:
for name, param in nn_conv.named_parameters():
    print(name, param.shape)
for name, param in nn_conv.named_parameters():
    print(name, param.shape)

### Output

In [ ]:
x = torch.randn(batch_size, in_channels, 5, 5)
nn_out = nn_conv(x)
out = conv(x)
print(f"{nn_out.shape}\n{nn_out=}")
print(f"{out.shape}\n{out=}")